In [ ]:
# For colab
import os
from google.colab import userdata

# 1. Pull the secrets using the interactive Colab kernel
colab_hf_token = userdata.get('HF_TOKEN')
colab_wandb_key = userdata.get('WANDB_API_KEY')

# 2. Inject them into the system environment variables
os.environ["HF_TOKEN"] = colab_hf_token
os.environ["WANDB_API_KEY"] = colab_wandb_key

print("✅ Secrets successfully injected into environment.")

In [ ]:
# Replace this:
!python parallel_hardware_trainer.py

In [ ]:
# With this:
import subprocess
import time
import os

MAX_RETRIES = 5
MIN_RUNTIME_SECONDS = 60   # if it dies in < 60s, count as a "fast crash" and back off harder
BASE_BACKOFF = 30          # seconds
USER_STOP_MARKER = "/tmp/USER_STOPPED_TRAINING"

# Clean any stale marker from a previous notebook run
if os.path.exists(USER_STOP_MARKER):
    os.remove(USER_STOP_MARKER)

attempt = 0
while attempt < MAX_RETRIES:
    attempt += 1
    print(f"\n{'='*19}\n🚀 Launch attempt {attempt}/{MAX_RETRIES}\n{'='*19}")
    
    start = time.time()
    result = subprocess.run(["python", "-u", "parallel_hardware_trainer.py"])
    runtime = time.time() - start
    
    # User hit Stop -> don't auto-restart
    if os.path.exists(USER_STOP_MARKER):
        print("🛑 User-initiated stop detected. Not restarting.")
        os.remove(USER_STOP_MARKER)
        break
    
    # Clean exit -> training finished or graceful end
    if result.returncode == 0:
        print(f"✅ Trainer exited cleanly after {runtime:.0f}s.")
        break
    
    # Crash
    print(f"💥 Trainer exited with code {result.returncode} after {runtime:.0f}s.")
    
    if attempt >= MAX_RETRIES:
        print(f"⚠️ Hit max retries ({MAX_RETRIES}). Giving up.")
        break
    
    # Fast crashes get exponential backoff (likely a deterministic bug or persistent
    # TPU lock issue). Long runs that crashed get short backoff (transient stall).
    if runtime < MIN_RUNTIME_SECONDS:
        backoff = BASE_BACKOFF * (2 ** (attempt - 1))
        print(f"⚠️ Fast crash. Backing off {backoff}s before retry...")
    else:
        backoff = BASE_BACKOFF
        print(f"⏳ Restarting in {backoff}s. (Resume from last checkpoint via training_state.json)")
    
    time.sleep(backoff)

print("\n🏁 Launcher done.")